In [ ]:
# BLOQUE 1: INSTALACIONES
!pip install prophet statsmodels plotly -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Prophet
from prophet import Prophet

# ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Métricas
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error)

# Gráficos
import plotly.graph_objects as go

print("✅ Todo instalado y listo")

✅ Todo instalado y listo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# BLOQUE 2: CREAR DATOS DE PRUEBA

np.random.seed(42)
fechas = pd.date_range(
    start='2024-01-01', periods=90, freq='D'
)

tendencia = 100 + np.arange(90) * 1.5
estacionalidad = 30 * np.sin(
    np.arange(90) * 2 * np.pi / 7
)
ruido = np.random.normal(0, 8, 90)
ventas = tendencia + estacionalidad + ruido

df = pd.DataFrame({
    'ds': fechas,
    'y': ventas
})

# División 70/30
split = int(len(df) * 0.70)
df_train = df[:split]
df_test = df[split:]

print(f"✅ Datos listos")
print(f"Entrenamiento: {len(df_train)} días")
print(f"Validación:    {len(df_test)} días")

✅ Datos listos
Entrenamiento: 62 días
Validación:    28 días


In [ ]:
# BLOQUE 3: MODELO PROPHET

def correr_prophet(df_train, df_test):
    """
    Entrena Prophet y devuelve métricas.
    """
    print("🔄 Probando Prophet...")

    # Entrenar
    modelo = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95
    )
    modelo.fit(df_train)

    # Predecir período de validación
    futuro = modelo.make_future_dataframe(
        periods=len(df_test), freq='D'
    )
    prediccion = modelo.predict(futuro)
    pred = prediccion['yhat'].tail(
        len(df_test)
    ).values
    real = df_test['y'].values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   Prophet MAPE: {mape:.2f}%")

    return {
        'nombre': 'Prophet',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': modelo,
        'prediccion': prediccion
    }

print("✅ Función Prophet lista")

✅ Función Prophet lista


In [ ]:
# BLOQUE 4: MODELO ARIMA

def correr_arima(df_train, df_test,
                 orden=(1,1,1)):
    """
    Entrena ARIMA y devuelve métricas.
    orden = (p, d, q)
    p = autoregresivo
    d = diferenciación
    q = media móvil
    """
    print("🔄 Probando ARIMA...")

    # Entrenar ARIMA
    modelo = ARIMA(
        df_train['y'],
        order=orden
    )
    resultado = modelo.fit()

    # Predecir período de validación
    predicciones = resultado.forecast(
        steps=len(df_test)
    )
    # forecast = predice N pasos futuros

    real = df_test['y'].values
    pred = predicciones.values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   ARIMA MAPE: {mape:.2f}%")

    # Crear predicción completa
    # (histórico + futuro)
    pred_completa = resultado.predict(
        start=0,
        end=len(df_train) + len(df_test) - 1
    )

    return {
        'nombre': f'ARIMA{orden}',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': resultado,
        'prediccion': pred_completa
    }

print("✅ Función ARIMA lista")

✅ Función ARIMA lista


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Prophet
from prophet import Prophet

# ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Métricas
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error)

# Gráficos
import plotly.graph_objects as go


def correr_prophet(df_train, df_test):
    """
    Entrena Prophet y devuelve métricas.
    """
    print("🔄 Probando Prophet...")

    # Entrenar
    modelo = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95
    )
    modelo.fit(df_train)

    # Predecir período de validación
    futuro = modelo.make_future_dataframe(
        periods=len(df_test), freq='D'
    )
    prediccion = modelo.predict(futuro)
    pred = prediccion['yhat'].tail(
        len(df_test)
    ).values
    real = df_test['y'].values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   Prophet MAPE: {mape:.2f}%")

    return {
        'nombre': 'Prophet',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': modelo,
        'prediccion': prediccion
    }


def correr_arima(df_train, df_test,
                 orden=(1,1,1)):
    """
    Entrena ARIMA y devuelve métricas.
    orden = (p, d, q)
    p = autoregresivo
    d = diferenciación
    q = media móvil
    """
    print("🔄 Probando ARIMA...")

    # Entrenar ARIMA
    modelo = ARIMA(
        df_train['y'],
        order=orden
    )
    resultado = modelo.fit()

    # Predecir período de validación
    predicciones = resultado.forecast(
        steps=len(df_test)
    )
    # forecast = predice N pasos futuros

    real = df_test['y'].values
    pred = predicciones.values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   ARIMA MAPE: {mape:.2f}%")

    # Crear predicción completa
    # (histórico + futuro)
    pred_completa = resultado.predict(
        start=0,
        end=len(df_train) + len(df_test) - 1
    )

    return {
        'nombre': f'ARIMA{orden}',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': resultado,
        'prediccion': pred_completa
    }


# BLOQUE 5: SISTEMA QUE COMPARA Y ELIGE MEJOR

def mejor_modelo(df, dias_futuro=30):
    """
    Prueba Prophet y ARIMA.
    Elige automáticamente el mejor.
    Devuelve predicción del ganador.
    """
    print("=" * 45)
    print("  COMPARANDO MODELOS...")
    print("=" * 45)

    # División 70/30
    split = int(len(df) * 0.70)
    df_train = df[:split]
    df_test = df[split:]

    # Correr ambos modelos
    resultado_prophet = correr_prophet(
        df_train, df_test
    )
    resultado_arima = correr_arima(
        df_train, df_test, orden=(1,1,1)
    )

    # Comparar resultados
    resultados = [resultado_prophet,
                  resultado_arima]

    # Ordenar por MAPE (menor = mejor)
    resultados.sort(key=lambda x: x['mape'])

    ganador = resultados[0]
    perdedor = resultados[1]

    print("\n" + "=" * 45)
    print("  RESULTADO COMPARACIÓN")
    print("=" * 45)
    print(f"  🥇 GANADOR:  {ganador['nombre']}")
    print(f"     MAPE:    {ganador['mape']}%")
    print(f"     MAE:     {ganador['mae']}")
    print(f"\n  🥈 Segundo: {perdedor['nombre']}")
    print(f"     MAPE:    {perdedor['mape']}%")
    print(f"     MAE:     {perdedor['mae']}")
    print("=" * 45)

    # Reentrenar ganador con TODOS los datos
    print(f"\n🔄 Reentrenando {ganador['nombre']} con datos completos...")

    if ganador['nombre'] == 'Prophet':
        modelo_final = Prophet(
            weekly_seasonality=True,
            yearly_seasonality=False,
            daily_seasonality=False,
            interval_width=0.95
        )
        modelo_final.fit(df)

        futuro = modelo_final.make_future_dataframe(
            periods=dias_futuro, freq='D'
        )
        prediccion_final = modelo_final.predict(futuro)

    else:
        modelo_final = ARIMA(
            df['y'], order=(1,1,1)
        )
        resultado_final = modelo_final.fit()

        # Crear DataFrame compatible
        pred_hist = resultado_final.predict(
            start=0, end=len(df)-1
        )
        pred_fut = resultado_final.forecast(
            steps=dias_futuro
        )

        # Combinar histórico + futuro
        fechas_futuras = pd.date_range(
            start=df['ds'].max(),
            periods=dias_futuro + 1,
            freq='D'
        )[1:]

        prediccion_final = pd.DataFrame({
            'ds': pd.concat([
                df['ds'],
                pd.Series(fechas_futuras)
            ]).reset_index(drop=True),
            'yhat': pd.concat([
                pred_hist,
                pred_fut
            ]).reset_index(drop=True)
        })

        # Agregar intervalos simples
        std = df['y'].std()
        prediccion_final['yhat_upper'] = (
            prediccion_final['yhat'] + 1.96 * std
        )
        prediccion_final['yhat_lower'] = (
            prediccion_final['yhat'] - 1.96 * std
        )

    print(f"✅ {ganador['nombre']} listo para usar")

    metricas = {
        'modelo_ganador': ganador['nombre'],
        'MAPE': ganador['mape'],
        'MAE': ganador['mae'],
        'Precision': round(100 - ganador['mape'], 2),
        'comparacion': {
            ganador['nombre']: ganador['mape'],
            perdedor['nombre']: perdedor['mape']
        }
    }

    return modelo_final, prediccion_final, metricas

In [ ]:
# BLOQUE 6: EJECUTAR COMPARACIÓN

# Correr sistema comparador
modelo, prediccion, metricas = mejor_modelo(
    df=df,
    dias_futuro=30
)

# Ver resultado final
print("\n📊 RESUMEN FINAL:")
print(f"Modelo elegido: {metricas['modelo_ganador']}")
print(f"Precisión:      {metricas['Precision']}%")
print(f"MAPE:           {metricas['MAPE']}%")
print(f"\nComparación completa:")
for modelo_nombre, mape in metricas['comparacion'].items():
    emoji = "🥇" if mape == metricas['MAPE'] else "🥈"
    print(f"{emoji} {modelo_nombre}: {mape}%")

  COMPARANDO MODELOS...
🔄 Probando Prophet...
   Prophet MAPE: 3.41%
🔄 Probando ARIMA...
   ARIMA MAPE: 23.68%

  RESULTADO COMPARACIÓN
  🥇 GANADOR:  Prophet
     MAPE:    3.41%
     MAE:     7.02

  🥈 Segundo: ARIMA(1, 1, 1)
     MAPE:    23.68%
     MAE:     53.03

🔄 Reentrenando Prophet con datos completos...
✅ Prophet listo para usar

📊 RESUMEN FINAL:
Modelo elegido: Prophet
Precisión:      96.59%
MAPE:           3.41%

Comparación completa:
🥇 Prophet: 3.41%
🥈 ARIMA(1, 1, 1): 23.68%


In [ ]:
# BLOQUE 7: INTEGRAR AL SCRIPT PRINCIPAL

# Reemplaza la función predecir() del dia5
# por mejor_modelo()

# PRUEBA FINAL:
df_resultado, prediccion, metricas = mejor_modelo(
    df=df,
    dias_futuro=30
)

print(f"\n✅ PRODUCTO FINAL:")
print(f"Modelo:    {metricas['modelo_ganador']}")
print(f"Precisión: {metricas['Precision']}% ")
print(f"MAPE:      {metricas['MAPE']}% ")

  COMPARANDO MODELOS...
🔄 Probando Prophet...
   Prophet MAPE: 3.41%
🔄 Probando ARIMA...
   ARIMA MAPE: 23.68%

  RESULTADO COMPARACIÓN
  🥇 GANADOR:  Prophet
     MAPE:    3.41%
     MAE:     7.02

  🥈 Segundo: ARIMA(1, 1, 1)
     MAPE:    23.68%
     MAE:     53.03

🔄 Reentrenando Prophet con datos completos...
✅ Prophet listo para usar

✅ PRODUCTO FINAL:
Modelo:    Prophet
Precisión: 96.59% 
MAPE:      3.41% 


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Prophet
from prophet import Prophet

# ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Métricas
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error)

# Gráficos
import plotly.graph_objects as go


def correr_prophet(df_train, df_test):
    """
    Entrena Prophet y devuelve métricas.
    """
    print("🔄 Probando Prophet...")

    # Entrenar
    modelo = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95
    )
    modelo.fit(df_train)

    # Predecir período de validación
    futuro = modelo.make_future_dataframe(
        periods=len(df_test), freq='D'
    )
    prediccion = modelo.predict(futuro)
    pred = prediccion['yhat'].tail(
        len(df_test)
    ).values
    real = df_test['y'].values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   Prophet MAPE: {mape:.2f}%")

    return {
        'nombre': 'Prophet',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': modelo,
        'prediccion': prediccion
    }


def correr_arima(df_train, df_test,
                 orden=(1,1,1)):
    """
    Entrena ARIMA y devuelve métricas.
    orden = (p, d, q)
    p = autoregresivo
    d = diferenciación
    q = media móvil
    """
    print("🔄 Probando ARIMA...")

    # Entrenar ARIMA
    modelo = ARIMA(
        df_train['y'],
        order=orden
    )
    resultado = modelo.fit()

    # Predecir período de validación
    predicciones = resultado.forecast(
        steps=len(df_test)
    )
    # forecast = predice N pasos futuros

    real = df_test['y'].values
    pred = predicciones.values

    # Métricas
    mape = np.mean(
        np.abs((real - pred) / real)
    ) * 100
    mae = mean_absolute_error(real, pred)

    print(f"   ARIMA MAPE: {mape:.2f}%")

    # Crear predicción completa
    # (histórico + futuro)
    pred_completa = resultado.predict(
        start=0,
        end=len(df_train) + len(df_test) - 1
    )

    return {
        'nombre': f'ARIMA{orden}',
        'mape': round(mape, 2),
        'mae': round(mae, 2),
        'modelo': resultado,
        'prediccion': pred_completa
    }


# BLOQUE 5: SISTEMA QUE COMPARA Y ELIGE MEJOR

def mejor_modelo(df, dias_futuro=30):
    """
    Prueba Prophet y ARIMA.
    Elige automáticamente el mejor.
    Devuelve predicción del ganador.
    """
    print("=" * 45)
    print("  COMPARANDO MODELOS...")
    print("=" * 45)

    # División 70/30
    split = int(len(df) * 0.70)
    df_train = df[:split]
    df_test = df[split:]

    # Correr ambos modelos
    resultado_prophet = correr_prophet(
        df_train, df_test
    )
    resultado_arima = correr_arima(
        df_train, df_test, orden=(1,1,1)
    )

    # Comparar resultados
    resultados = [resultado_prophet,
                  resultado_arima]

    # Ordenar por MAPE (menor = mejor)
    resultados.sort(key=lambda x: x['mape'])

    ganador = resultados[0]
    perdedor = resultados[1]

    print("\n" + "=" * 45)
    print("  RESULTADO COMPARACIÓN")
    print("=" * 45)
    print(f"  🥇 GANADOR:  {ganador['nombre']}")
    print(f"     MAPE:    {ganador['mape']}%")
    print(f"     MAE:     {ganador['mae']}")
    print(f"\n  🥈 Segundo: {perdedor['nombre']}")
    print(f"     MAPE:    {perdedor['mape']}%")
    print(f"     MAE:     {perdedor['mae']}")
    print("=" * 45)

    # Reentrenar ganador con TODOS los datos
    print(f"\n🔄 Reentrenando {ganador['nombre']} con datos completos...")

    if ganador['nombre'] == 'Prophet':
        modelo_final = Prophet(
            weekly_seasonality=True,
            yearly_seasonality=False,
            daily_seasonality=False,
            interval_width=0.95
        )
        modelo_final.fit(df)

        futuro = modelo_final.make_future_dataframe(
            periods=dias_futuro, freq='D'
        )
        prediccion_final = modelo_final.predict(futuro)

    else:
        modelo_final = ARIMA(
            df['y'], order=(1,1,1)
        )
        resultado_final = modelo_final.fit()

        # Crear DataFrame compatible
        pred_hist = resultado_final.predict(
            start=0,
            end=len(df)-1
        )
        pred_fut = resultado_final.forecast(
            steps=dias_futuro
        )

        # Combinar histórico + futuro
        fechas_futuras = pd.date_range(
            start=df['ds'].max(),
            periods=dias_futuro + 1,
            freq='D'
        )[1:]

        prediccion_final = pd.DataFrame({
            'ds': pd.concat([
                df['ds'],
                pd.Series(fechas_futuras)
            ]).reset_index(drop=True),
            'yhat': pd.concat([
                pred_hist,
                pred_fut
            ]).reset_index(drop=True)
        })

        # Agregar intervalos simples
        std = df['y'].std()
        prediccion_final['yhat_upper'] = (
            prediccion_final['yhat'] + 1.96 * std
        )
        prediccion_final['yhat_lower'] = (
            prediccion_final['yhat'] - 1.96 * std
        )

    print(f"✅ {ganador['nombre']} listo para usar")

    metricas = {
        'modelo_ganador': ganador['nombre'],
        'MAPE': ganador['mape'],
        'MAE': ganador['mae'],
        'Precision': round(100 - ganador['mape'], 2),
        'comparacion': {
            ganador['nombre']: ganador['mape'],
            perdedor['nombre']: perdedor['mape']
        }
    }

    return modelo_final, prediccion_final, metricas

# BLOQUE 7: SCRIPT COMPLETO VERSIÓN 2.0
# Une todo: limpieza + comparador + dashboard

def analizar_negocio_v2(datos,
                         nombre_negocio="Mi Negocio",
                         dias_futuro=30):
    """
    VERSIÓN 2.0 DEL SCRIPT PRINCIPAL

    Mejoras vs versión 1.0:
    - Compara Prophet vs ARIMA automáticamente
    - Elige el mejor modelo
    - Dashboard muestra qué modelo ganó
    """
    print("=" * 45)
    print(f"  ANÁLISIS V2.0: {nombre_negocio}")
    print("=" * 45)

    # PASO 1: Limpiar datos
    print("\n🔄 Limpiando datos...")
    df = datos.copy()
    df['ds'] = pd.to_datetime(df['ds'])
    df = df.sort_values('ds').reset_index(drop=True)
    df = df.dropna(subset=['ds', 'y'])
    df = df.drop_duplicates(subset=['ds'])
    df['y'] = pd.to_numeric(df['y'], errors='coerce')
    df = df[df['y'] >= 0]
    print(f"✅ Datos limpios: {len(df)} registros")

    # PASO 2: Comparar modelos
    modelo, prediccion, metricas = mejor_modelo(
        df=df,
        dias_futuro=dias_futuro
    )

    # PASO 3: Generar dashboard v2
    print("\n🔄 Generando dashboard v2...")

    # Gráfico 1: Predicción principal
    fig1 = go.Figure()

    fig1.add_trace(go.Scatter(
        x=df['ds'], y=df['y'],
        name='Ventas reales',
        line=dict(color='#2196F3', width=2),
        mode='lines+markers',
        marker=dict(size=4)
    ))

    fig1.add_trace(go.Scatter(
        x=prediccion['ds'],
        y=prediccion['yhat'],
        name=f'Predicción ({metricas["modelo_ganador"]})',
        line=dict(color='#FF5722', width=2, dash='dash')
    ))

    fig1.add_trace(go.Scatter(
        x=pd.concat([prediccion['ds'],
                     prediccion['ds'][::-1]]),
        y=pd.concat([prediccion['yhat_upper'],
                     prediccion['yhat_lower'][::-1]]),
        fill='toself',
        fillcolor='rgba(255,87,34,0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Intervalo 95%'
    ))

    # Add the vertical line using the Timestamp directly
    fig1.add_vline(
        x=df['ds'].max(),
        line_dash='dot',
        line_color='green',
        line_width=2
    )

    # Add the annotation separately
    fig1.add_annotation(
        x=df['ds'].max(),
        y=1,  # Y-coordinate (relative to paper height)
        xref='x',  # X-coordinate refers to the 'x' axis
        yref='paper',  # Y-coordinate refers to the paper (0 to 1)
        text='Hoy',
        showarrow=False,
        xanchor='right', # Position text to the right of the line
        yanchor='top', # Position text at the top of the plot
        font=dict(color='green', size=12)
    )

    fig1.update_layout(
        title=dict(
            text=f'{nombre_negocio} - Predicción de Ventas',
            font=dict(size=18), x=0.5
        ),
        xaxis_title='Fecha',
        yaxis_title='Ventas',
        template='plotly_white',
        hovermode='x unified',
        height=500
    )

    # Gráfico 2: Comparación de modelos
    modelos_nombres = list(
        metricas['comparacion'].keys()
    )
    modelos_mape = list(
        metricas['comparacion'].values()
    )

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=modelos_nombres,
        y=modelos_mape,
        marker_color=[
            '#4CAF50' if m == min(modelos_mape)
            else '#FF5722'
            for m in modelos_mape
        ],
        # Verde = ganador, Rojo = perdedor
        text=[f'{m}%' for m in modelos_mape],
        textposition='outside'
    ))

    fig2.update_layout(
        title=dict(
            text='Comparación de Modelos (menor MAPE = mejor)',
            font=dict(size=16), x=0.5
        ),
        xaxis_title='Modelo',
        yaxis_title='MAPE (%)',
        template='plotly_white',
        height=350
    )

    # Gráfico 3: Tendencia
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=prediccion['ds'],
        y=prediccion['trend'],
        name='Tendencia',
        line=dict(color='#4CAF50', width=3)
    ))

    fig3.update_layout(
        title=dict(
            text=f'Tendencia de Crecimiento - {nombre_negocio}',
            font=dict(size=16), x=0.5
        ),
        xaxis_title='Fecha',
        yaxis_title='Ventas',
        template='plotly_white',
        height=400
    )

    # PASO 4: Tabla de predicciones
    pred_futuras = prediccion[
        prediccion['ds'] > df['ds'].max()
    ][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head(30)

    pred_futuras.columns = [
        'Fecha', 'Predicción', 'Mínimo', 'Máximo'
    ]
    pred_futuras['Fecha'] = pred_futuras[
        'Fecha'
    ].dt.strftime('%d/%m/%Y')
    pred_futuras = pred_futuras.round(2)

    # Convertir tabla a HTML
    tabla_html = pred_futuras.to_html(
        index=False,
        classes='tabla-predicciones',
        border=0
    )

    # Mejor día
    weekly = prediccion[['ds', 'weekly']].copy()
    weekly['dia'] = weekly['ds'].dt.dayofweek
    weekly_avg = weekly.groupby('dia')['weekly'].mean()
    dias_nombres = ['Lunes', 'Martes', 'Miércoles',
                    'Jueves', 'Viernes', 'Sábado', 'Domingo']
    mejor_dia = dias_nombres[weekly_avg.idxmax()]

    # PASO 5: Exportar HTML
    nombre_archivo = (
        f'/content/dashboard_v2_'
        f'{nombre_negocio.replace(" ", "_")}.html'
    )

    with open(nombre_archivo, 'w',
              encoding='utf-8') as f:
        f.write(f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="utf-8">
    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">
    <title>Dashboard v2 - {nombre_negocio}</title>
    <style>
        * {{ margin:0; padding:0; box-sizing:border-box; }}
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            background: #F0F2F5;
            color: #333;
        }}
        .header {{
            background: linear-gradient(135deg, #1565C0, #42A5F5);
            color: white;
            padding: 30px 20px;
            text-align: center;
        }}
        .header h1 {{ font-size: 24px; font-weight: 700; }}
        .header h2 {{ font-size: 16px; opacity: 0.9; margin-top: 8px; }}
        .header p {{ font-size: 12px; opacity: 0.7; margin-top: 5px; }}
        .badge {{
            display: inline-block;
            background: rgba(255,255,255,0.2);
            padding: 5px 15px;
            border-radius: 20px;
            font-size: 13px;
            margin-top: 10px;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            padding: 25px 15px;
        }}
        .metricas {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 15px;
            margin-bottom: 25px;
        }}
        .metrica-card {{
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            text-align: center;
            border-top: 4px solid #1565C0;
        }}
        .metrica-icono {{ font-size: 26px; margin-bottom: 8px; }}
        .metrica-valor {{
            font-size: 24px;
            font-weight: 700;
            color: #1565C0;
        }}
        .metrica-label {{
            font-size: 11px;
            color: #888;
            margin-top: 5px;
            text-transform: uppercase;
        }}
        .alerta {{
            background: #E3F2FD;
            border-left: 4px solid #1565C0;
            border-radius: 8px;
            padding: 15px 20px;
            margin-bottom: 20px;
            font-size: 14px;
            color: #1565C0;
        }}
        .seccion-titulo {{
            font-size: 15px;
            font-weight: 600;
            color: #444;
            margin-bottom: 12px;
            padding-left: 10px;
            border-left: 4px solid #1565C0;
        }}
        .grafico {{
            background: white;
            border-radius: 12px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        }}
        .tabla-predicciones {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
        }}
        .tabla-predicciones th {{
            background: #1565C0;
            color: white;
            padding: 10px;
            text-align: center;
        }}
        .tabla-predicciones td {{
            padding: 8px 10px;
            text-align: center;
            border-bottom: 1px solid #eee;
        }}
        .tabla-predicciones tr:nth-child(even) {{
            background: #F5F5F5;
        }}
        .footer {{
            text-align: center;
            color: #aaa;
            font-size: 12px;
            padding: 20px;
            border-top: 1px solid #ddd;
        }}
        @media (max-width: 600px) {{
            .metricas {{ grid-template-columns: repeat(2, 1fr); }}
        }}
    </style>
</head>
<body>

<div class="header">
    <h1>📊 Dashboard Predictivo de Ventas</h1>
    <h2>{nombre_negocio}</h2>
    <p>Análisis automático con IA</p>
    <div class="badge">
        🤖 Modelo seleccionado: {metricas['modelo_ganador']}
    </div>
</div>

<div class="container">

    <div class="metricas">
        <div class="metrica-card">
            <div class="metrica-icono">🎯</div>
            <div class="metrica-valor">{metricas['Precision']}%</div>
            <div class="metrica-label">Precisión</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-icono">📉</div>
            <div class="metrica-valor">{metricas['MAPE']}%</div>
            <div class="metrica-label">Error promedio</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-icono">⭐</div>
            <div class="metrica-valor">{mejor_dia}</div>
            <div class="metrica-label">Mejor día</div>
        </div>
        <div class="metrica-card">
            <div class="metrica-icono">📅</div>
            <div class="metrica-valor">{len(df)}</div>
            <div class="metrica-label">Días analizados</div>
        </div>
    </div>

    <div class="alerta">
        💡 <strong>Resumen ejecutivo:</strong>
        Tu negocio muestra tendencia creciente.
        Mejor día de venta: <strong>{mejor_dia}</strong>.
        Modelo <strong>{metricas['modelo_ganador']}</strong>
        seleccionado automáticamente con
        <strong>{metricas['Precision']}%</strong> de precisión.
    </div>

    <p class="seccion-titulo">Predicción de Ventas</p>
    <div class="grafico">
        {fig1.to_html(full_html=False,
                      include_plotlyjs='cdn')}
    </div>

    <p class="seccion-titulo">Comparación de Modelos</p>
    <div class="grafico">
        {fig2.to_html(full_html=False,
                      include_plotlyjs=False)}
    </div>

    <p class="seccion-titulo">Tendencia de Crecimiento</p>
    <div class="grafico">
        {fig3.to_html(full_html=False,
                      include_plotlyjs=False)}
    </div>

    <p class="seccion-titulo">
        Predicciones próximos 30 días
    </p>
    <div class="grafico">
        {tabla_html}
    </div>

</div>

<div class="footer">
    <p>Dashboard Predictivo v2.0 • Prophet + ARIMA + Python</p>
    <p>Modelo entrenado con {len(df)} días de datos</p>
</div>

</body>
</html>
        """
        )

    print(f"✅ Dashboard v2 guardado")
    print(f"📁 {nombre_archivo}")

    return df, prediccion, metricas

print("✅ Script v2.0 completo listo")

✅ Script v2.0 completo listo


In [ ]:
# BLOQUE 8: EJECUTAR SCRIPT COMPLETO V2.0

# Crear datos de ejemplo
np.random.seed(42)
fechas = pd.date_range(
    start='2024-01-01', periods=90, freq='D'
)
tendencia = 100 + np.arange(90) * 1.5
estacionalidad = 30 * np.sin(
    np.arange(90) * 2 * np.pi / 7
)
ruido = np.random.normal(0, 8, 90)
ventas = tendencia + estacionalidad + ruido

datos_tienda = pd.DataFrame({
    'ds': fechas,
    'y': ventas
})

# EJECUTAR V2.0
df, prediccion, metricas = analizar_negocio_v2(
    datos=datos_tienda,
    nombre_negocio="Tienda El Alto",
    dias_futuro=30
)

  ANÁLISIS V2.0: Tienda El Alto

🔄 Limpiando datos...
✅ Datos limpios: 90 registros
  COMPARANDO MODELOS...
🔄 Probando Prophet...
   Prophet MAPE: 3.41%
🔄 Probando ARIMA...
   ARIMA MAPE: 23.68%

  RESULTADO COMPARACIÓN
  🥇 GANADOR:  Prophet
     MAPE:    3.41%
     MAE:     7.02

  🥈 Segundo: ARIMA(1, 1, 1)
     MAPE:    23.68%
     MAE:     53.03

🔄 Reentrenando Prophet con datos completos...
✅ Prophet listo para usar

🔄 Generando dashboard v2...
✅ Dashboard v2 guardado
📁 /content/dashboard_v2_Tienda_El_Alto.html
